# Sistemas Recomendadores de Películas

En este notebook construiremos tres tipos de sistemas de recomendación:

1. Recomendador simple basado en media ponderada (estilo IMDb Top 250)
2. Recomendador basado en contenido usando TF-IDF sobre sinopsis
3. Recomendador basado en metadatos (cast, director, keywords, géneros)

Dataset: MovieLens + TMDB (~45,000 películas)

import pandas as pd
import numpy as np

# Cargar metadatos principales
df = pd.read_csv('../data/movies_metadata.csv', low_memory=False)

df.head(3)

df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce')
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce')

df_clean = df.dropna(subset=['vote_count', 'vote_average'])

C = df_clean['vote_average'].mean()
m = df_clean['vote_count'].quantile(0.90)

C, m

q_movies = df_clean.copy().loc[df_clean['vote_count'] >= m]
q_movies.shape

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

## Recomendador simple (estilo IMDb Top 250)

Usamos la fórmula de media ponderada para calcular una puntuación más justa para cada película:

\[
\text{score} = \left(\frac{v}{v+m} \cdot R\right) + \left(\frac{m}{v+m} \cdot C\right)
\]

Donde:
- `R` = calificación promedio de la película
- `v` = número de votos
- `C` = promedio global de calificaciones
- `m` = mínimo de votos requerido para ser considerado

## Resultados

Las siguientes son las 15 películas con mejor puntuación ponderada según el modelo IMDb.  
Este ranking combina calidad (calificación promedio) y popularidad (cantidad de votos).

In [7]:
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)
q_movies = q_movies.sort_values('score', ascending=False)

q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171
